# Notebook 06 — Prediction Models
### WID2003 Cognitive Science | FSKTM, Universiti Malaya

---

## Overview

This notebook reduces a redundant 416-feature space to a de-duplicated set, tunes five classifiers and a soft-voting ensemble to predict High vs. Low performance from eye-tracking features, and evaluates the winner on a genuinely held-out test set. Feature selection is performed **inside** the cross-validation loop to prevent data leakage, and the held-out test split is kept fully separate from every model-selection step — see Background below.

**Input**
| File | Description |
|---|---|
| `data/processed/dataset_final.parquet` | Wide-format feature matrix with performance labels |

**Outputs**
| File | Description |
|---|---|
| `outputs/models/trained_model.pkl` | Best model (single tuned pipeline or soft-voting ensemble) refitted on full dataset |
| `outputs/models/model_metadata.json` | Model name, feature-reduction provenance, tuning record, CV scores, decision threshold |
| `outputs/reports/classification_report.csv` | Per-model untuned CV results (screening pass) |
| `outputs/figures/06_*.png` | Model comparison bar chart, permutation test, threshold tuning curve, confusion matrix, ROC curve |

---

## Learning Objectives

By the end of this notebook, you should be able to:

1. Explain why **Leave-One-Out cross-validation (LOO-CV)** is appropriate for small samples (N < 30), why **Stratified k-Fold CV** is preferred once N is larger, and how the two differ
2. Describe the **pipeline pattern** in scikit-learn — why wrapping feature selection and the classifier into one `Pipeline` object prevents data leakage
3. Interpret a **permutation test** result and explain what it means when p > 0.05
4. Distinguish **accuracy** from **balanced accuracy** and explain when each metric is appropriate
5. Interpret a **ROC-AUC** score and a **confusion matrix** together to characterise a classifier's behaviour
6. Compare multiple classifiers on the same evaluation protocol and justify which model should be preferred
7. Explain two independent techniques for handling class imbalance — **`class_weight='balanced'`** (changes what the model learns) and **threshold tuning** (changes how predictions are read off a fixed model) — and why using held-out test data to tune either would itself be a form of leakage
8. Explain why two features correlated at r = 1.0 are *structurally* redundant, and why univariate feature selection (`SelectKBest`) has no way to notice on its own
9. Contrast `f_classif` and `mutual_info_classif` as feature-scoring functions, and describe a relationship one can detect that the other cannot
10. Describe how a soft-voting ensemble combines member predictions, and explain why ensembling *tuned* components gives a different (and more meaningful) result than ensembling components at default settings

---

## Background

### Cross-Validation Strategy

With a small sample (20–30 participants), a standard 80/20 train-test split leaves very few test examples. **Cross-validation** instead uses all data for both training and evaluation by rotating the hold-out partition.

| N | Strategy | Why |
|---|---|---|
| ≥ 30 | Stratified 5-Fold CV | Efficient; each fold tests ~20% of data |
| < 30 | Leave-One-Out CV (LOO) | Each participant is the test set exactly once; uses maximum training data per fold |

LOO-CV produces N evaluation folds (one per participant). The reported score is the mean across all folds.

This study now has **130 participants**, split into 104 train / 26 test by Notebook 04. `N` in the code below refers to the **training** count (104), which is what determines the CV strategy (`N >= MIN_N_FOR_KFOLD`) — so the code takes the **Stratified 5-Fold CV** branch, not LOO-CV. The LOO-CV path still exists and runs automatically if this notebook is ever re-run on a smaller pilot dataset.

### Keeping the Test Split Genuinely Held Out

An earlier version of this notebook ran `GridSearchCV` and `permutation_test_score` on **all 130 participants** — including the 26 rows Notebook 04 had already set aside as the test split. The final "hold-out" evaluation then refit the chosen model on the 104 training rows and tested on the 26 test rows, but because the model *architecture and hyperparameters* had already been selected using CV folds that included those same 26 participants, the resulting hold-out accuracy was optimistic, not a clean estimate. (See `outputs/reports/07_interpretation_report.md` §6 for the original writeup of this issue.)

This version fixes it: `## 1. Load data` below splits `X`/`y` into `X_train`/`y_train` and `X_test`/`y_test` immediately, and **every** model-selection step — feature-space reduction, cross-validating all 5 models, `GridSearchCV`, the ensemble, the permutation test, and threshold tuning — runs only on `X_train`/`y_train`. `X_test`/`y_test` are touched exactly once, in `## 8`, for the final evaluation.

### Reducing the Feature Space

`## 1b` reduces the 416-column feature matrix before any model sees it. The motivation: many features are near-perfect duplicates of each other (e.g. `{task}_distractor_sum_X` and `{task}_distractor_mean_X` correlate at exactly r = 1.0000 for every task, because each task has exactly one distractor object) — `SelectKBest` is univariate and has no way to notice this, so it can waste several of its `k` slots on redundant copies of the same signal instead of genuinely different features. `## 1b` groups mutually-correlated features (train-only Spearman correlation, `|r| > 0.95`) via a union-find/connected-components approach and keeps only the most label-relevant member of each group (train-only ANOVA F-score). Everything in `## 1b` is computed from `X_train` only — see the "why train-only" and "why not use Notebook 05's FDR list" boxes there for why that matters.

### The Pipeline Pattern and Data Leakage

**Data leakage** occurs when information from the test fold influences the training process. A common mistake:

```python
# WRONG — leaky:
selector.fit(X, y)                  # sees the whole dataset including the test fold
X_selected = selector.transform(X)
cross_val_score(clf, X_selected, y)

# CORRECT — leak-free:
pipe = Pipeline([('select', selector), ('clf', clf)])
cross_val_score(pipe, X, y)         # selector is refit inside each fold
```

By wrapping both steps in a `Pipeline`, scikit-learn ensures the `SelectKBest` is refit on training data only within each cross-validation fold.

### Classifiers Used

| Model | Key hyperparameters | Why included |
|---|---|---|
| `DummyClassifier` | `strategy='most_frequent'` | Baseline — shows what pure chance achieves |
| Logistic Regression | `C` (inverse regularisation), `penalty` (L1/L2) | Interpretable linear model; coefficient magnitudes indicate feature importance |
| Decision Tree | `max_depth` | Interpretable tree structure; prone to overfitting with small N |
| Random Forest | `n_estimators`, `max_depth` | Ensemble reduces overfitting; provides built-in feature importance |
| SVM (RBF kernel) | `C`, `gamma` | Effective in high-dimensional, small-N settings |

All four (excluding the dummy baseline) are also given `class_weight='balanced'` — see "Class Imbalance" below. A soft-voting ensemble of the top 3 *tuned* models is also considered — see "Tuning Every Candidate" below and `## 6b`.

### Evaluation Metrics

| Metric | Formula | Use case |
|---|---|---|
| **Accuracy** | (TP + TN) / N | Misleading for imbalanced classes |
| **Balanced accuracy** | (sensitivity + specificity) / 2 | Accounts for class imbalance; preferred here |
| **F1-macro** | Mean of per-class F1 | Balances precision and recall across both classes |
| **ROC-AUC** | Area under ROC curve | Measures separability; 0.5 = chance, 1.0 = perfect |

With a near-balanced median split, all four metrics should be reported, but **balanced accuracy** is the primary metric for model selection.

### Class Imbalance: `class_weight='balanced'` and Threshold Tuning

The training data is imbalanced (High=77, Low=53 in the full 130; roughly the same ratio in the 104-row training split). Earlier runs of this pipeline showed the resulting models were consistently better at recalling Low performers than High performers — the model was "conservative" about predicting High. Two complementary fixes are applied here:

1. **`class_weight='balanced'`** (in `## 3. Define models`) changes *what the model learns*: it reweights the training loss inversely to class frequency, so misclassifying the minority class (Low) costs more during fitting. This affects the model's coefficients/splits themselves.
2. **Threshold tuning** (`## 7b`) changes *how a fixed, already-trained model's probabilities are converted into a Low/High call*. Instead of the default 0.5 cutoff on `predict_proba`, it scans a grid of thresholds against **out-of-fold predictions on the training set** (via `cross_val_predict`) and picks the one that maximises balanced accuracy.

Both techniques must only ever see training data — tuning either one against the test set would leak test information into the "held-out" evaluation, the exact problem described above.

### Tuning Every Candidate, Not Just the Untuned Winner

`## 6a` grid-searches **all four** non-dummy models, not just whichever one happens to score best at default hyperparameters in `## 4`/`## 5`. Default hyperparameters are arbitrary, so the untuned ranking is an unreliable proxy for tuned potential — especially right after a change like the `## 1b` feature reduction, which can shift which model's inductive bias fits the new feature set best. `## 6a` also widens `select__k` (was `[5,10,15]`, now up to `30`) and adds `mutual_info_classif` alongside `f_classif` as an alternative feature-scoring function. `## 6b` then builds a soft-voting ensemble from the **tuned** top-3 models and makes the final selection among all 5 tuned finalists.

### Permutation Test

A **permutation test** checks whether the model's CV score is genuinely above chance:

1. Shuffle the labels 1000 times
2. Rerun cross-validation on each shuffled version → null distribution
3. Compute p-value = proportion of permuted scores ≥ observed score

If p > 0.05, the model's performance cannot be distinguished from random label assignment. With small N, this is common and should be reported honestly.

### Nested Cross-Validation for Hyperparameter Tuning

Hyperparameter tuning via `GridSearchCV` introduces its own selection bias — the "best" parameters appear to perform better partly by fitting the CV folds they were selected on. **Nested CV** uses an outer CV loop for performance estimation and an inner loop for hyperparameter search, providing an unbiased estimate. Nested CV is always more computationally expensive than a single loop (it multiplies the number of model fits by the number of outer folds); the single-loop approach used here is acceptable with honest reporting — and, since the training split (104 rows) is now never contaminated by the true test rows, the single true hold-out evaluation in `## 8` still provides an unbiased check on top of it, even without going as far as full nested CV. Tuning 4 model families and then selecting among 5 finalists (the 4 singles plus the `## 6b` ensemble) on the same folds is itself a mild version of this same optimism — one more reason the `## 8` hold-out number, which was never used to select anything, is the number to trust most.

---

## Discussion Questions

1. With N = 25 participants, LOO-CV produces 25 evaluation folds. In each fold, the model trains on 24 participants and tests on 1. What are the advantages and disadvantages of this strategy compared to 5-fold CV? Now that this study has N = 130 (104 train / 26 test), which strategy does the code actually use, and why does that answer change with sample size?
2. A Random Forest achieves 85% balanced accuracy in cross-validation but only 60% on the hold-out test set. What are three possible explanations for this gap, and what would you do to investigate each one? (Hint: one explanation is exactly the bug this notebook used to have — see "Keeping the Test Split Genuinely Held Out" above.)
3. The permutation test returns p = 0.18. Does this mean the model is useless? How would you report this result in a study write-up? (This study's actual permutation test came back p = 0.001 — how should that change the confidence with which findings are reported?)
4. Accuracy is 80% and balanced accuracy is 61% for the same model on the same data. How is this possible? What does it tell you about the class distribution and the model's behaviour?
5. You are comparing Logistic Regression (balanced accuracy = 0.72 ± 0.15) and Random Forest (balanced accuracy = 0.74 ± 0.22). Which would you select as the final model, and why? What additional information would help you decide?
6. `class_weight='balanced'` and threshold tuning both address class imbalance, but at different points in the pipeline (training vs. prediction time). Give a scenario where you'd want only one of the two, and one where you'd want both together.
7. `T1_Prisoner-15sec_distractor_sum_Number_of_Visits` and `..._mean_Number_of_Visits` correlate at exactly r = 1.0000. Why is this structural rather than coincidental? Before `## 1b`, `SelectKBest(k=10)` spent 4 of its 10 slots on such duplicate pairs — what does that cost the model?
8. `## 1b` uses a correlation threshold of 0.95. What would change at 0.90 (more aggressive) or 0.99 (more conservative)? Which mistake is worse in this context: discarding a feature that carried some independent signal, or keeping a near-duplicate?
9. Union-find clustering is transitive: if feature A correlates with B at 0.96, and B correlates with C at 0.96, all three end up in one cluster even if A and C only correlate at 0.80 — so C could be dropped in favor of A even though C isn't really redundant with A. Is this an acceptable trade-off? What would a stricter clustering method (e.g. complete-linkage) do differently, and at what cost?
10. Notebook 05 found 59 FDR-significant features using all 130 participants. Why is it *not* legitimate to filter down to those 59 features inside this notebook, even though Notebook 05's statistical test (Mann-Whitney + FDR correction) is arguably more rigorous than the ANOVA F-test used inside `SelectKBest`?
11. Suppose the `## 6b` ensemble scores lower balanced accuracy than the best single model, but higher ROC-AUC. Which would you deploy in a tool meant to flag students for follow-up support, and why? How does `## 7b`'s threshold tuning change your answer?
12. This notebook now searches 4 model families × up to 6 `k` values × 2 score functions × several classifier-specific hyperparameters, then picks the best of 5 finalists — all evaluated on the same 5 CV folds. How does this inflate the reported CV score of the winner? Which single number in this notebook is *not* inflated by this process, and why?

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

%cd /content/drive/MyDrive/WID2003

!python -m pip install -q -r requirements.txt

# Important: notebook imports assume the working directory is notebooks/
%cd /content/drive/MyDrive/WID2003/notebooks

In [ ]:
import sys
sys.path.insert(0, '..')

import json
import warnings
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
warnings.filterwarnings('ignore')

from functools import partial
from sklearn.base           import clone
from sklearn.dummy            import DummyClassifier
from sklearn.linear_model     import LogisticRegression
from sklearn.tree             import DecisionTreeClassifier
from sklearn.ensemble         import RandomForestClassifier, VotingClassifier
from sklearn.svm              import SVC
from sklearn.feature_selection import SelectKBest, f_classif, mutual_info_classif
from sklearn.model_selection  import (
    StratifiedKFold, LeaveOneOut, cross_validate, cross_val_predict,
    GridSearchCV, permutation_test_score
)
from sklearn.metrics          import (
    classification_report, confusion_matrix,
    roc_auc_score, roc_curve, RocCurveDisplay,
    ConfusionMatrixDisplay, balanced_accuracy_score
)
from sklearn.pipeline         import Pipeline

from src.config import (
    DATASET_FINAL, OUTPUTS_MODELS, OUTPUTS_REPORTS, OUTPUTS_FIGURES,
    PERFORMANCE_LABEL_COL, RANDOM_STATE, MIN_N_FOR_KFOLD, CV_FOLDS_LARGE
)

sns.set_theme(style='whitegrid')
for p in [OUTPUTS_MODELS, OUTPUTS_REPORTS, OUTPUTS_FIGURES]:
    p.mkdir(parents=True, exist_ok=True)

# mutual_info_classif estimates mutual information with a kNN-based method that
# adds small random noise to break ties; pin the seed so grid-search results
# (## 6a) are reproducible across reruns.
mutual_info_seeded = partial(mutual_info_classif, random_state=RANDOM_STATE)
mutual_info_seeded.__name__ = 'mutual_info_classif'  # readable in best_params_

## 1. Load data

In [ ]:
dataset = pd.read_parquet(DATASET_FINAL)

# Identify scaled feature columns (exclude unscaled copies, metadata, labels)
exclude_cols = [
    'participant_id', PERFORMANCE_LABEL_COL, 'speed_label',
    'accuracy_rate', 'total_score', 'split', 'mean_response_time_ms'
]
task_correct_cols = [c for c in dataset.columns if c.endswith('_correct') and len(c) < 30]
exclude_cols += task_correct_cols
unscaled_cols  = [c for c in dataset.columns if c.endswith('_unscaled')]
exclude_cols  += unscaled_cols

feature_cols = [c for c in dataset.columns if c not in exclude_cols]

print(f"Total participants: {len(dataset)}")
print(f"Feature columns:    {len(feature_cols)}")

X = dataset[feature_cols].values
y = dataset[PERFORMANCE_LABEL_COL].astype(int).values

print(f"Class distribution: High={y.sum()}, Low={(y==0).sum()}")

# Model selection, hyperparameter tuning, and the permutation test below must
# never see the held-out test rows, or the final "hold-out" evaluation isn't a
# clean estimate. (This was a real bug in an earlier version of this notebook:
# GridSearchCV and permutation_test_score both ran on the full 130 rows, which
# already included the 26 test participants set aside by Notebook 04's split
# — see outputs/reports/07_interpretation_report.md §6 for the writeup.)
# Notebook 04's stratified split (`dataset['split']`) is reused here rather
# than re-split, since the feature scaler was already fit on those same rows.
has_holdout = 'split' in dataset.columns and (dataset['split'] == 'test').sum() > 0
if has_holdout:
    train_mask = (dataset['split'] == 'train').values
    test_mask  = (dataset['split'] == 'test').values
    X_train, y_train = X[train_mask], y[train_mask]
    X_test,  y_test  = X[test_mask],  y[test_mask]
    print(f"\nTrain: {len(y_train)} participants (High={y_train.sum()}, Low={(y_train==0).sum()}) "
          f"— used for all model selection, tuning, and the permutation test below")
    print(f"Test:  {len(y_test)} participants (High={y_test.sum()}, Low={(y_test==0).sum()}) "
          f"— held out untouched until the final evaluation (## 8)")
else:
    print("\nNo hold-out split available (small N) — using all data for CV; "
          "no independent test-set evaluation is possible in this run.")
    X_train, y_train = X, y
    X_test, y_test = None, None

N = len(y_train)

## 1b. Feature-space reduction (train-only)

`feature_cols` above has **416 columns for 104 training participants** — a p/n ratio of roughly 4:1. `SelectKBest` is univariate: it scores each feature against the label independently, with no notion of redundancy between features. That has a real, measurable cost.

### Proof this matters: the model already shipped with duplicate slots

The Decision Tree currently saved to `outputs/models/trained_model.pkl` used `SelectKBest(k=10)`, which picked:

```
Toothbrush_correct_Total_duration_of_Glances       ┐
Toothbrush_correct_Total_duration_of_Visit         ├ r ≥ 0.9965 — one signal, three slots
Toothbrush_correct_Total_duration_of_fixations     ┘
T1_Prisoner-15sec_distractor_mean_Number_of_Visits            ┐ r = 1.0000
T1_Prisoner-15sec_distractor_sum_Number_of_Visits             ┘
T1_Prisoner-15sec_distractor_mean_Total_duration_of_fixations ┐ r = 1.0000
T1_Prisoner-15sec_distractor_sum_Total_duration_of_fixations  ┘
T5_Fish-15sec_distractor_mean_Number_of_Visits
mean_fixation_duration_across_tasks
total_correct_aoi_dwell_ratio
```

**Only 6 of the 10 selected features carry distinct information.** The other 4 slots are wasted on near-perfect duplicates of features already selected.

### Why the duplication is structural, not incidental

- `{task}_distractor_sum_X` and `{task}_distractor_mean_X` are **exactly** correlated (r = 1.0000) for every task and every metric checked — because each task in this study has exactly one distractor object, `sum` and `mean` over one value are numerically identical.
- `{task}_correct_Total_duration_of_fixations`, `_of_Visit`, and `_of_Glances` measure overlapping time windows within the same AOI (r ≥ 0.9664 in every task checked; `Visit`↔`Glances` is tightest, ≥ 0.9965 always).

### Algorithm (train-only — see box below)

1. Drop features that are constant within the 104-row training split (their `f_classif` F-score is undefined — NaN — which the pipeline was silently discarding via `warnings.filterwarnings('ignore')` above).
2. Compute a Spearman correlation matrix among the remaining features, using **`X_train` only**.
3. Compute each feature's univariate ANOVA F-score against `y_train` (`f_classif`, train-only) — used only as a tie-breaker.
4. Build a graph: an edge between two features whenever `|correlation| > CORR_THRESHOLD`. Find its connected components with a union-find (disjoint-set) structure — each component is a cluster of mutually-redundant features.
5. Within each cluster, keep only the feature with the highest F-score; drop the rest.

> **Why train-only?** This notebook was restructured earlier to keep the 26-participant test split fully out of every model-selection step (see `## 1`'s "Keeping the Test Split Genuinely Held Out"). Computing the correlation matrix or the tie-break F-scores on the full 130-row dataset would let the test participants influence *which columns even exist* going into `SelectKBest` — a leak of exactly the same class already fixed once in this notebook.

> **Why not just use Notebook 05's 59 FDR-significant features as the shortlist?** `outputs/reports/05_mann_whitney_results.csv` is a legitimate, more rigorous statistical test (Mann-Whitney U + Benjamini-Hochberg FDR correction) — but it was computed on **all 130 participants**, including the 26 held out here. Filtering on it would leak test-set label information into feature selection. The overlap is reported below purely for narrative interest; it is not used in any computation.

> **A chaining caveat, honestly stated:** connected components are transitive. If A–B correlate at 0.96 and B–C at 0.96, all three merge into one cluster even if A–C is only 0.80 — so C gets dropped in favor of whichever of A/B/C has the best F-score, even though C isn't actually redundant with A. This is deliberately conservative. A stricter alternative (complete-linkage clustering, guaranteeing *every* pair within a cluster exceeds the threshold) exists in `scipy.cluster.hierarchy` but is not used here to keep the algorithm traceable in ~15 lines.

Note honestly: even after this step, the surviving feature count is still much larger than 104 — deduplication does not fix the p/n ratio by itself. What it fixes is **wasted `SelectKBest` slots on near-duplicate ties**, as demonstrated above. The p/n ratio itself is addressed by `select__k` staying small relative to the feature count (`## 6a`).

In [ ]:
# ── Feature-space reduction ──────────────────────────────────────────────────
# Everything in this cell is computed from X_train ONLY. The correlation matrix
# and the F-scores must never see X_test, or the choice of which columns exist
# would itself be informed by the held-out participants.
CORR_THRESHOLD = 0.95

train_df = pd.DataFrame(X_train, columns=feature_cols)

# Step 0 — drop features that are constant within the training split.
# f_classif divides by within-group variance, so constant columns yield NaN
# F-scores (silently discarded so far by warnings.filterwarnings('ignore')).
constant_cols = [c for c in feature_cols if train_df[c].nunique() <= 1]
kept_cols     = [c for c in feature_cols if c not in constant_cols]
train_df      = train_df[kept_cols]
print(f"Dropped {len(constant_cols)} features constant in the training split "
      f"({len(feature_cols)} -> {len(kept_cols)})")

# Step 1 — univariate ANOVA F-score per feature, train-only. Used only as the
# within-cluster tie-breaker: of several interchangeable columns, keep the one
# most related to the label.
f_scores = np.nan_to_num(f_classif(train_df.values, y_train)[0], nan=0.0)

# Step 2 — Spearman correlation matrix (rank-based: robust to the monotone but
# non-linear relationships common in duration/count eye-tracking features).
corr = np.abs(np.nan_to_num(train_df.corr(method='spearman').values, nan=0.0))
np.fill_diagonal(corr, 0.0)

# Step 3 — group mutually-correlated features into clusters with union-find
# (disjoint-set). Each feature starts in its own set; every pair above the
# threshold merges its two sets. What remains are the connected components of
# the correlation graph.
n_feat = len(kept_cols)
parent = list(range(n_feat))

def find(a):
    """Root of a's set, with path compression."""
    while parent[a] != a:
        parent[a] = parent[parent[a]]   # point a straight at its grandparent
        a = parent[a]
    return a

def union(a, b):
    """Merge the sets containing a and b."""
    root_a, root_b = find(a), find(b)
    if root_a != root_b:
        parent[root_b] = root_a

rows, cols = np.where(np.triu(corr, k=1) > CORR_THRESHOLD)
for i, j in zip(rows, cols):
    union(int(i), int(j))

clusters = {}
for idx in range(n_feat):
    clusters.setdefault(find(idx), []).append(idx)

# Step 4 — keep the highest-F member of each cluster, drop the rest.
keep_idx = sorted(max(members, key=lambda m: f_scores[m])
                  for members in clusters.values())

feature_cols_full = feature_cols                 # keep the original list for the record
feature_cols      = [kept_cols[i] for i in keep_idx]

# Step 5 — rebuild every matrix from the surviving columns. X is rebuilt too,
# because ## 9 refits on the full dataset before saving, and Notebook 07
# reads feature_col_names out of the metadata written there.
X       = dataset[feature_cols].values
X_train = X[train_mask] if has_holdout else X
X_test  = X[test_mask]  if has_holdout else None

multi = {r: m for r, m in clusters.items() if len(m) > 1}
print(f"\nCorrelation clustering at |Spearman r| > {CORR_THRESHOLD}:")
print(f"  clusters with >1 member: {len(multi)}")
print(f"  largest cluster:         {max((len(m) for m in multi.values()), default=1)} features")
print(f"  features: {len(feature_cols_full)} -> {len(feature_cols)} "
      f"({len(feature_cols_full) - len(feature_cols)} removed)")

print("\nThree largest clusters (kept feature marked *):")
for members in sorted(multi.values(), key=len, reverse=True)[:3]:
    winner = max(members, key=lambda m: f_scores[m])
    for m in sorted(members, key=lambda m: -f_scores[m]):
        print(f"  {'*' if m == winner else ' '} F={f_scores[m]:6.2f}  {kept_cols[m]}")
    print()

# ── Informational only: overlap with Notebook 05's FDR-significant features ──
# NOT used as a filter. Notebook 05 ran Mann-Whitney on all 130 participants,
# including the 26 held-out test rows, so selecting on `significant_fdr` here
# would leak test-set label information into feature selection — exactly the
# bug this notebook was restructured to eliminate. Reported for narrative
# interest only; nothing below depends on it.
mw_path = OUTPUTS_REPORTS / '05_mann_whitney_results.csv'
fdr_sig = set()
if mw_path.exists():
    mw = pd.read_csv(mw_path)
    fdr_sig = set(mw.loc[mw['significant_fdr'], 'feature'])
    print(f"Notebook 05 FDR-significant features (all 130 rows): {len(fdr_sig)}")
    print(f"  ...still present after deduplication:              {len(fdr_sig & set(feature_cols))}")
    print("  (informational only — not used as a selection filter, see markdown above)")

## 2. Cross-validation strategy

In [ ]:
if N < MIN_N_FOR_KFOLD:
    print(f"N={N} < {MIN_N_FOR_KFOLD} — using Leave-One-Out CV")
    cv = LeaveOneOut()
    cv_name = 'LOO-CV'
else:
    k = CV_FOLDS_LARGE
    print(f"N={N} >= {MIN_N_FOR_KFOLD} — using Stratified {k}-Fold CV")
    cv = StratifiedKFold(n_splits=k, shuffle=True, random_state=RANDOM_STATE)
    cv_name = f'Stratified {k}-Fold CV'

## 3. Define models

In [ ]:
# Feature selection: SelectKBest inside pipeline to prevent leakage
K_FEATURES = min(10, len(feature_cols))  # untuned starting point; ## 6a searches over k

def make_pipeline(classifier):
    return Pipeline([
        ('select', SelectKBest(f_classif, k=K_FEATURES)),
        ('clf',    classifier)
    ])

# class_weight='balanced' reweights each class inversely to its frequency
# (High=77, Low=53 in the training data) so the loss function penalises
# errors on the minority class (Low) more heavily. This directly targets the
# recall asymmetry documented in outputs/reports/07_interpretation_report.md
# §6 — without it, the model was consistently better at recalling Low
# performers than High performers.
#
# NOTE: a soft-voting ensemble is deliberately NOT defined here. Ensembling
# four models at these arbitrary default hyperparameters measures very
# little — an ensemble of all four untuned pipelines scores only 0.695
# balanced accuracy, worse than three of its own members once tuned. The
# ensemble is instead built in ## 6b from the *tuned* pipelines that come
# out of ## 6a.
models = {
    'Dummy (baseline)': make_pipeline(
        DummyClassifier(strategy='most_frequent', random_state=RANDOM_STATE)
    ),
    'Logistic Regression': make_pipeline(
        LogisticRegression(max_iter=1000, random_state=RANDOM_STATE, C=1.0,
                            class_weight='balanced')
    ),
    'Decision Tree': make_pipeline(
        DecisionTreeClassifier(max_depth=4, random_state=RANDOM_STATE,
                                class_weight='balanced')
    ),
    'Random Forest': make_pipeline(
        RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE,
                                class_weight='balanced')
    ),
    'SVM (RBF)': make_pipeline(
        SVC(kernel='rbf', probability=True, random_state=RANDOM_STATE,
            class_weight='balanced')
    ),
}

## 4. Cross-validate all models

In [ ]:
# ROC-AUC requires >=2 samples in the test fold (one per class).
# With LOO-CV each test fold has exactly 1 sample, so roc_auc is undefined → NaN.
if cv_name == 'LOO-CV':
    scoring = ['accuracy', 'balanced_accuracy', 'f1_macro']
    print("NOTE: roc_auc excluded from LOO-CV scoring (undefined for 1-sample test folds).")
else:
    scoring = ['accuracy', 'balanced_accuracy', 'f1_macro', 'roc_auc']

cv_results = {}

for name, model in models.items():
    print(f"Evaluating: {name}...")
    result = cross_validate(model, X_train, y_train, cv=cv, scoring=scoring,
                            return_train_score=False, n_jobs=-1)
    cv_results[name] = result

print("\nDone.")

In [ ]:
summary_rows = []
for name, result in cv_results.items():
    row = {'Model': name}
    for metric in scoring:
        scores = result[f'test_{metric}']
        row[f'{metric}_mean'] = scores.mean().round(3)
        row[f'{metric}_std']  = scores.std().round(3)
    summary_rows.append(row)

summary_df = pd.DataFrame(summary_rows).set_index('Model')
summary_df.to_csv(OUTPUTS_REPORTS / 'classification_report.csv')
print(f"CV Strategy: {cv_name}")
summary_df

## 5. Visualize CV results

This is a *screening* pass — every model here is still at its default hyperparameters. The real model selection happens in `## 6a`, after each model has had a chance to be tuned; default hyperparameters can be a misleading proxy for a model's actual best-case performance (see `## 6a` for a concrete example of this reshuffling the ranking).

In [ ]:
has_roc = 'roc_auc_mean' in summary_df.columns
n_plots = 2 if has_roc else 1
fig, axes = plt.subplots(1, n_plots, figsize=(7 * n_plots, 5))
if n_plots == 1:
    axes = [axes]

colors = ['#bdc3c7'] + ['#3498db'] * (len(summary_df) - 1)

# Balanced accuracy comparison
means = summary_df['balanced_accuracy_mean']
stds  = summary_df['balanced_accuracy_std']
axes[0].barh(means.index, means.values, xerr=stds.values,
             color=colors, edgecolor='white', capsize=4)
axes[0].axvline(0.5, color='red', linestyle='--', label='Chance level')
axes[0].set_xlabel('Balanced Accuracy')
axes[0].set_title(f'Balanced Accuracy ({cv_name})')
axes[0].legend()
axes[0].set_xlim(0, 1)

# ROC-AUC comparison (only when available)
if has_roc:
    means_roc = summary_df['roc_auc_mean']
    stds_roc  = summary_df['roc_auc_std']
    axes[1].barh(means_roc.index, means_roc.values, xerr=stds_roc.values,
                 color=colors, edgecolor='white', capsize=4)
    axes[1].axvline(0.5, color='red', linestyle='--', label='Chance level')
    axes[1].set_xlabel('ROC-AUC')
    axes[1].set_title(f'ROC-AUC ({cv_name})')
    axes[1].legend()
    axes[1].set_xlim(0, 1)
else:
    print("ROC-AUC plot skipped (not computed under LOO-CV).")

plt.tight_layout()
plt.savefig(OUTPUTS_FIGURES / '06_model_comparison.png', dpi=150)
plt.show()

## 6a. Hyperparameter tuning (all candidate models)

### Why tune all four, not just the untuned CV winner

The previous version of this notebook grid-searched only the single best model from `## 4`'s untuned comparison. That's a bad idea once the feature set changes: default hyperparameters are arbitrary, so the untuned ranking (`## 4`/`## 5` above) is a poor proxy for tuned potential — which model looks best at default settings and which model has the best *tuned* ceiling are not reliably the same model, especially right after a change like the feature deduplication in `## 1b`. Grid-searching only the untuned winner risks tuning the wrong model entirely and never discovering that a different candidate would have tuned to a higher score.

So every non-dummy candidate below is tuned, and the final choice (`## 6b`) is made on **tuned** scores, not the `## 4` screening-pass scores.

### What was widened

- `select__k`: was `[5, 10, 15]`; now `[5, 10, 15, 20, 25, 30]` (capped at the post-deduplication feature count from `## 1b`) — the earlier grid never tried using more than 15 of the ~240 surviving features.
- `select__score_func`: new — alternates between `f_classif` (ANOVA F-test: fast, exact, but only detects *linear/monotone* separation between class means) and `mutual_info_classif` (kNN-based mutual information: slower, approximate, but detects *any* statistical dependence, including relationships where both very high and very low feature values indicate the same class — something `f_classif` scores as near-zero).

### Honesty about multiple comparisons

Tuning 4 model families and then picking the best of 5 finalists (4 singles + the `## 6b` ensemble) on the same 5 CV folds is itself a mild form of the same selection-bias problem the "Nested Cross-Validation" section above already discusses — the reported CV score of the winner is somewhat optimistic. The untouched `## 8` hold-out evaluation is what keeps this honest; that number was never used to pick anything.

Measured cost on a comparable machine (24 cores, `n_jobs=-1`): roughly 30 seconds for all four grid searches combined.

In [ ]:
# Candidate k values, capped at the post-deduplication feature count.
K_GRID = [k for k in [5, 10, 15, 20, 25, 30] if k <= len(feature_cols)]

# Both scoring functions are searched for every model:
#   f_classif            — ANOVA F-test; fast, exact, but only detects
#                          differences in class means (monotone/linear signal).
#   mutual_info_classif  — kNN-based mutual information; slower, approximate,
#                          but detects any statistical dependence, including
#                          non-monotone relationships f_classif scores as ~0.
SCORE_FUNCS = [f_classif, mutual_info_seeded]

param_grids = {
    'Logistic Regression': {
        'select__k':          K_GRID,
        'select__score_func': SCORE_FUNCS,
        'clf__C':              [0.01, 0.1, 1.0, 10.0],
        'clf__penalty':        ['l1', 'l2'],
        'clf__solver':         ['liblinear'],
    },
    'Decision Tree': {
        'select__k':              K_GRID,
        'select__score_func':     SCORE_FUNCS,
        'clf__max_depth':         [2, 3, 4, 5],
        'clf__min_samples_split': [2, 4],
    },
    'Random Forest': {
        'select__k':          K_GRID,
        'select__score_func': SCORE_FUNCS,
        'clf__n_estimators':  [50, 100, 200],
        'clf__max_depth':     [None, 3, 5],
    },
    'SVM (RBF)': {
        'select__k':          K_GRID,
        'select__score_func': SCORE_FUNCS,
        'clf__C':              [0.1, 1.0, 10.0],
        'clf__gamma':          ['scale', 'auto'],
    },
}

# Tune EVERY candidate, not just the untuned CV winner (see markdown above for
# why). All tuning uses X_train/y_train only.
tuned_models, tuned_scores, tuned_params = {}, {}, {}

for name, grid in param_grids.items():
    n_combos = int(np.prod([len(v) for v in grid.values()]))
    print(f"GridSearchCV: {name} ({n_combos} combinations x {cv.get_n_splits()} folds)...")
    gs = GridSearchCV(models[name], grid, cv=cv,
                      scoring='balanced_accuracy', n_jobs=-1, refit=True)
    gs.fit(X_train, y_train)
    tuned_models[name] = gs.best_estimator_
    tuned_scores[name] = gs.best_score_
    tuned_params[name] = gs.best_params_
    print(f"  best CV balanced_accuracy = {gs.best_score_:.3f}")
    print(f"  best params = {gs.best_params_}\n")

tuned_df = (pd.DataFrame({
        'untuned_cv': summary_df['balanced_accuracy_mean'].drop('Dummy (baseline)'),
        'tuned_cv':   pd.Series(tuned_scores),
    })
    .assign(gain=lambda d: (d['tuned_cv'] - d['untuned_cv']).round(3))
    .sort_values('tuned_cv', ascending=False))
print("Tuned vs untuned CV balanced accuracy (train split only):")
tuned_df

## 6b. Soft-voting ensemble and final model selection

**Soft voting** combines several classifiers by averaging their `predict_proba` outputs and taking the argmax — as opposed to *hard* voting, which only looks at each member's final Low/High label and takes a majority. With as few as 3 members, soft voting matters more than hard voting because it lets a confident member outweigh two uncertain ones, rather than being outvoted 2-to-1.

The ensemble's members are the **tuned** pipelines from `## 6a` — not fresh default-hyperparameter pipelines. Ensembling untuned models measures very little (a dry run of "all 4 at defaults" scored only 0.695, worse than 3 of its own members once those members are tuned); ensembling already-tuned pipelines is the standard, defensible construction. Because each member was tuned independently, they may end up with different `select__k` and different `select__score_func` — the ensemble genuinely benefits from members that each look at the data a different way.

There is deliberately **no `param_grids` entry for the ensemble itself**. Its members are already individually tuned; searching a joint grid over 3 sub-pipelines' hyperparameters would multiply combinatorially (roughly 10⁵–10⁶ configurations) for no clear benefit, and would make the notebook far harder to follow.

**A trade-off to expect**: adding weaker members to the ensemble tends to raise ROC-AUC (a ranking metric — averaging several probability estimates usually improves how well participants are *ordered* from most-to-least-likely-High) while balanced accuracy at the default threshold can move either direction (it depends on the ranking *and* on where exactly 0.5 falls relative to the averaged probabilities — the same issue `## 7b`'s threshold tuning exists to address). If the ensemble under-performs the best single model on balanced accuracy but wins on AUC, that is a genuine trade-off, not a bug — see Discussion Question 11.

In [ ]:
ENSEMBLE_TOP_N = 3   # members = the N highest tuned-CV models

# VotingClassifier estimator names must be plain identifiers (no '__').
SLUGS = {'Logistic Regression': 'lr', 'Decision Tree': 'dt',
         'Random Forest': 'rf',       'SVM (RBF)': 'svm'}

ranked                 = tuned_df.index.tolist()
ensemble_member_names  = ranked[:ENSEMBLE_TOP_N]
print(f"Ensemble members (top {ENSEMBLE_TOP_N} by tuned CV): {ensemble_member_names}\n")

# clone() so the ensemble owns unfitted copies and the standalone tuned models
# in `tuned_models` are left untouched for the comparison below.
ensemble = VotingClassifier(
    estimators=[(SLUGS[n], clone(tuned_models[n])) for n in ensemble_member_names],
    voting='soft',
)

ens_cv = cross_validate(ensemble, X_train, y_train, cv=cv,
                        scoring=scoring, n_jobs=-1)
ENSEMBLE_NAME = f'Ensemble (Soft Voting, top-{ENSEMBLE_TOP_N})'
ens_score = ens_cv['test_balanced_accuracy'].mean()
print(f"{ENSEMBLE_NAME}: CV balanced accuracy = {ens_score:.3f}")
if 'test_roc_auc' in ens_cv:
    print(f"{ENSEMBLE_NAME}: CV ROC-AUC           = {ens_cv['test_roc_auc'].mean():.3f}")

# ── Final selection: 4 tuned singles + 1 ensemble, all scored on X_train ─────
final_candidates = dict(tuned_scores)
final_candidates[ENSEMBLE_NAME] = ens_score

best_name  = max(final_candidates, key=final_candidates.get)
best_model = clone(ensemble) if best_name == ENSEMBLE_NAME else tuned_models[best_name]
best_cv_score = final_candidates[best_name]

print("\nFinal candidate ranking (CV balanced accuracy, train split only):")
for n, s in sorted(final_candidates.items(), key=lambda kv: -kv[1]):
    print(f"  {'->' if n == best_name else '  '} {n:34s} {s:.3f}")
print(f"\nSelected: {best_name} (CV balanced accuracy = {best_cv_score:.3f})")
if best_name in tuned_params:
    print(f"Tuned params: {tuned_params[best_name]}")

best_model.fit(X_train, y_train)   # ## 7 / ## 7b below expect a fitted estimator

## 7. Permutation test (validate above-chance performance)

In [ ]:
print("Running permutation test (1000 permutations)...")
score, perm_scores, p_value = permutation_test_score(
    best_model, X_train, y_train,
    scoring='balanced_accuracy',
    cv=cv, n_permutations=1000,
    random_state=RANDOM_STATE, n_jobs=-1
)

print(f"Model CV score:      {score:.3f}")
print(f"Permutation p-value: {p_value:.4f}")
if p_value < 0.05:
    print("Result is significant — performance is above chance.")
else:
    print("WARNING: Performance is NOT significantly above chance. Interpret results cautiously.")

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(perm_scores, bins=25, color='steelblue', alpha=0.7, label='Permuted scores')
ax.axvline(score, color='red', linewidth=2, label=f'Observed score={score:.3f}')
ax.set_xlabel('Balanced Accuracy')
ax.set_title(f'Permutation Test — {best_name} (p={p_value:.4f})')
ax.legend()
plt.tight_layout()
plt.savefig(OUTPUTS_FIGURES / '06_permutation_test.png', dpi=150)
plt.show()

## 7b. Threshold tuning (fix recall asymmetry)

`class_weight='balanced'` (above) changes how the model *learns* — it reweights the training loss. Threshold tuning is a complementary, post-hoc fix: instead of the default 0.5 cutoff on `predict_proba`, scan a range of thresholds and pick the one that maximises balanced accuracy on **out-of-fold predictions from the training set only** (`cross_val_predict`, using the same CV split as everywhere else in this notebook). Never tune the threshold against the test set — that would be the same kind of leakage this notebook now avoids everywhere else.

Threshold tuning matters *more* when `best_model` is the `## 6b` soft-voting ensemble than for a single model: averaging several members' probabilities pulls the resulting values toward the centre of the [0, 1] range, so the "natural" cutoff between classes often isn't at 0.5 anymore.

In [ ]:
oof_proba = cross_val_predict(best_model, X_train, y_train, cv=cv,
                               method='predict_proba', n_jobs=-1)[:, 1]

thresholds = np.linspace(0.1, 0.9, 81)
threshold_scores = [balanced_accuracy_score(y_train, (oof_proba >= t).astype(int))
                     for t in thresholds]
best_threshold = float(thresholds[int(np.argmax(threshold_scores))])
default_score  = balanced_accuracy_score(y_train, (oof_proba >= 0.5).astype(int))
tuned_score    = max(threshold_scores)

print(f"Default threshold (0.5):        balanced accuracy = {default_score:.3f}")
print(f"Tuned threshold ({best_threshold:.2f}):        balanced accuracy = {tuned_score:.3f}")

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(thresholds, threshold_scores, color='steelblue')
ax.axvline(0.5, color='gray', linestyle='--', label=f'Default (0.5) = {default_score:.3f}')
ax.axvline(best_threshold, color='red', linestyle='--', label=f'Tuned ({best_threshold:.2f}) = {tuned_score:.3f}')
ax.set_xlabel('Classification threshold')
ax.set_ylabel('Balanced accuracy (out-of-fold, train set)')
ax.set_title('Threshold Tuning — Out-of-Fold CV Predictions')
ax.legend()
plt.tight_layout()
plt.savefig(OUTPUTS_FIGURES / '06_threshold_tuning.png', dpi=150)
plt.show()

## 8. Confusion matrix on hold-out test set (if available)

In [ ]:
if has_holdout:
    # X_train/y_train/X_test/y_test already come from the split made in ## 1
    # (never touched by CV, GridSearchCV, the permutation test, or threshold
    # tuning above) — this is the first and only time X_test/y_test are used.
    best_model.fit(X_train, y_train)
    y_proba_test = best_model.predict_proba(X_test)[:, 1] if hasattr(best_model, 'predict_proba') else None
    y_pred_default = best_model.predict(X_test)
    y_pred_tuned = (y_proba_test >= best_threshold).astype(int) if y_proba_test is not None else y_pred_default

    print("Classification report (hold-out test set, default 0.5 threshold):")
    print(classification_report(y_test, y_pred_default, target_names=['Low','High']))

    print(f"\nClassification report (hold-out test set, tuned threshold={best_threshold:.2f}):")
    print(classification_report(y_test, y_pred_tuned, target_names=['Low','High']))

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    # Confusion matrix (tuned threshold — the operating point this notebook
    # actually recommends, per the threshold tuning step above)
    ConfusionMatrixDisplay.from_predictions(
        y_test, y_pred_tuned, display_labels=['Low','High'], ax=axes[0], colorbar=False
    )
    axes[0].set_title(f'Confusion Matrix — {best_name} (tuned threshold={best_threshold:.2f})')

    # ROC curve (threshold-independent — shows the full trade-off curve)
    if y_proba_test is not None:
        RocCurveDisplay.from_predictions(y_test, y_proba_test, ax=axes[1], name=best_name)
        axes[1].plot([0,1],[0,1],'k--')
        axes[1].set_title('ROC Curve')

    plt.tight_layout()
    plt.savefig(OUTPUTS_FIGURES / '06_confusion_roc.png', dpi=150)
    plt.show()
else:
    print("No hold-out test set — cross-validation results are the primary evaluation.")

## 9. Save best model and metadata

In [ ]:
# Refit on all data (train + test) before saving — once the honest hold-out
# evaluation above is in hand, using every available row maximizes the
# quality of the model actually shipped for downstream use (Notebook 07).
best_model.fit(X, y)
joblib.dump(best_model, OUTPUTS_MODELS / 'trained_model.pkl')

# Also save every individually-tuned model from ## 6a (not just the winner) so
# Notebook 07 can compare feature importance / coefficients across all 4
# model families, not only the one that happened to win. These are the
# ## 6a versions — fit on X_train (104 rows) only, not refit on the full 130
# like best_model above, so a winner's importances shown side-by-side with
# its peers here may differ very slightly from its "official" shipped values.
joblib.dump(tuned_models, OUTPUTS_MODELS / 'tuned_models.pkl')

# k / score_func come off the fitted estimator, not the K_FEATURES default —
# GridSearchCV may have chosen a different k than the untuned starting point.
if isinstance(best_model, Pipeline):
    _sel = best_model.named_steps['select']
    k_selected       = int(_sel.k)
    score_func_used  = getattr(_sel.score_func, '__name__', str(_sel.score_func))
    ensemble_members = None
else:   # VotingClassifier — each member has its own selector
    k_selected, score_func_used = None, None
    ensemble_members = {
        nm: {'k': int(est.named_steps['select'].k),
             'score_func': getattr(est.named_steps['select'].score_func, '__name__', '?'),
             'clf': type(est.named_steps['clf']).__name__}
        for nm, est in best_model.named_estimators_.items()
    }

metadata = {
    'best_model_name':      best_name,
    'is_ensemble':           not isinstance(best_model, Pipeline),
    'ensemble_members':      ensemble_members,
    'cv_strategy':          cv_name,
    'n_participants_total': int(len(y)),
    'n_participants_train': int(N),
    'n_participants_test':  int(len(y_test)) if has_holdout else 0,
    # ── feature-reduction provenance (## 1b) ──
    'n_features_before_reduction': len(feature_cols_full),
    'n_features_constant_dropped': len(constant_cols),
    'n_features_after_reduction':  len(feature_cols),
    'correlation_threshold':       CORR_THRESHOLD,
    'fdr_significant_overlap':     len(fdr_sig & set(feature_cols)) if fdr_sig else None,
    'n_features_total':     len(feature_cols),
    'k_features_selected':  k_selected,
    'select_score_func':    score_func_used,
    'feature_col_names':    feature_cols,
    # ── tuning record (## 6a / ## 6b) ──
    'tuned_cv_scores':      {k: float(v) for k, v in tuned_scores.items()},
    'tuned_best_params':    {k: {p: str(v) for p, v in d.items()} for k, d in tuned_params.items()},
    'ensemble_cv_score':    float(ens_score),
    'best_model_tuned_cv_score': float(best_cv_score),
    # ── final evaluation ──
    'cv_balanced_accuracy': float(score),
    'permutation_p_value':  float(p_value),
    'decision_threshold':   best_threshold,
    'threshold_tuned_balanced_accuracy_oof': float(tuned_score),
    'threshold_default_balanced_accuracy_oof': float(default_score),
    'all_model_results':    summary_df.to_dict(),
}

with open(OUTPUTS_MODELS / 'model_metadata.json', 'w') as f:
    json.dump(metadata, f, indent=2)

print(f"Saved: {OUTPUTS_MODELS / 'trained_model.pkl'}")
print(f"Saved: {OUTPUTS_MODELS / 'tuned_models.pkl'} ({len(tuned_models)} models: {list(tuned_models)})")
print(f"Saved: {OUTPUTS_MODELS / 'model_metadata.json'}")
print(f"Recommended decision threshold: {best_threshold:.2f} (default 0.5 was not re-optimized for class balance)")